Supplementary Code S10
Permutation-based robustness diagnostics under fixed S9 training split
Purpose

This notebook implements a permutation-based diagnostic analysis to assess the statistical robustness and non-spuriousness of model performance observed in Supplementary Code S9.
The analysis is explicitly confirmatory and diagnostic: it evaluates whether observed discrimination exceeds what would be expected under random label assignment when all modeling choices are held fixed.

S10 does not define new models and does not re-estimate generalization performance.

Relationship to prior scripts

S8 establishes a leakage-aware internal validation baseline using nested cross-validation.

S9 introduces a locked test set and reports one-time deployment-like performance.

S10 evaluates whether the observed training-set performance structure is robust to label permutation under the same validation protocol.

Importantly, S10 reuses the training subset defined in S9 and never accesses the locked test data.

Key principles

Uses exactly the same target definition, feature preprocessing, and governed feature sets as in S9.

Restricts all analyses to the S9 training subset; locked test data are excluded.

Fixes the outer cross-validation splits across true-label and permuted-label runs.

Repeats the full nested cross-validation pipeline under random label permutations.

Interprets results as robustness diagnostics, not as performance estimates.

Methods implemented in S10

For each feature-set variant (FULL, CLINICAL, BIOMARKERS), the notebook performs:

Fixed training subset and validation protocol

The training subset defined in S9 is used without modification.
Outer cross-validation splits are generated once and reused across all permutation runs to control variance.

True-label baseline evaluation

Nested cross-validation is performed using the true outcome labels to obtain a reference distribution of performance metrics.

Permutation-based diagnostics

Outcome labels are randomly permuted multiple times, and the identical nested cross-validation procedure is repeated for each permutation.
This yields empirical null distributions for discrimination metrics (e.g., ROC AUC, MCC).

Statistical comparison

Observed performance under true labels is compared against the permutation-derived null distributions to compute one-sided permutation p-values.

Inputs (from S8–S9 workflow)

Required:

processed_dataset.csv

features_used_FULL.csv

features_used_CLINICAL.csv

features_used_BIOMARKERS.csv

S9_locked_split_indices.csv

Optional:

audit_nonnumeric_features.csv

All inputs are consistent with those used in S9, ensuring that permutation diagnostics are performed under identical preprocessing and feature governance assumptions.

Outputs

Per-variant tables:

S10_<VARIANT>__TRUE.csv — nested cross-validation metrics under true labels.

S10_<VARIANT>__PERM.csv — nested cross-validation metrics under permuted labels.

S10_<VARIANT>__ALL.csv — combined true and permuted results.

S10_<VARIANT>__SUMMARY.csv — permutation-based summary statistics and p-values.

Cross-variant summaries:

S10_MASTER__TRUE.csv

S10_MASTER__PERM.csv

S10_MASTER__ALL.csv

S10_MASTER__SUMMARY.csv

Audit log:

S10_run_log.json — full record of configuration, inputs, and outputs.

Interpretation note

Supplementary Code S10 provides a formal robustness check against chance-level discrimination and hidden data artifacts.
Significant separation between true-label performance and permutation-based null distributions supports the conclusion that observed predictive signal is not solely driven by random structure or overfitting.
These diagnostics complement, but do not replace, external validation and should be interpreted within the methodological scope of the study.

## Imports + configuration of tracks and outputs

In [1]:
from __future__ import annotations

import json
import platform
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, matthews_corrcoef


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Skipping features without any observed values.*")

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()

OUT_DIR = ROOT / "results" / "S10_permutation_diagnostic"
TAB_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"
for d in (TAB_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Writing outputs to:", OUT_DIR.resolve())

SEARCH_ROOTS = [
    ROOT,
    ROOT / "results",
    ROOT / "results" / "S2_nestedcv",
    ROOT / "results" / "S6_leakage_audit",
    ROOT / "results" / "S8_revised_models",
    ROOT / "results" / "S9_locked_test",
    Path("/mnt/data"),
    ROOT / "drive",
    ROOT / "drive" / "MyDrive",
]

def find_file(filename: str, roots: list[Path]) -> Path:
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.rglob(filename))
            if hits:
                hits = sorted(hits, key=lambda x: len(str(x)))
                return hits[0]
    raise FileNotFoundError(
        f"Missing required file: {filename}\nSearched in:\n - " + "\n - ".join(map(str, roots))
    )

def read_csv_required(name: str) -> tuple[pd.DataFrame, Path]:
    p = find_file(name, SEARCH_ROOTS)
    df = pd.read_csv(p)
    print(f"Loaded: {p} | shape={df.shape}")
    return df, p


ROOT: /content
Writing outputs to: /content/results/S10_permutation_diagnostic


## Loading inputs (processed dataset, feature lists, split S9, audit)

In [2]:
USE_TRAIN_ONLY_FROM_S9_SPLIT = True

df_raw, path_processed = read_csv_required("processed_dataset.csv")

p_full = find_file("features_used_FULL.csv", SEARCH_ROOTS)
p_clin = find_file("features_used_CLINICAL.csv", SEARCH_ROOTS)
p_bio  = find_file("features_used_BIOMARKERS.csv", SEARCH_ROOTS)

S9_SPLIT_PATH = None
if USE_TRAIN_ONLY_FROM_S9_SPLIT:
    S9_SPLIT_PATH = find_file("S9_locked_split_indices.csv", SEARCH_ROOTS)

try:
    audit_df, path_audit = read_csv_required("audit_nonnumeric_features.csv")
except FileNotFoundError:
    audit_df, path_audit = None, None

print("Feature lists:")
print(" -", p_full)
print(" -", p_clin)
print(" -", p_bio)
print("S9 split:", S9_SPLIT_PATH)
print("Audit nonnumeric:", path_audit)

def load_feature_list(path: Path) -> list[str]:
    fdf = pd.read_csv(path)
    fdf.columns = [c.strip() for c in fdf.columns]
    for col in ["feature", "features", "feature_name", "column", "variable", "var", "name"]:
        if col in fdf.columns:
            feats = fdf[col].astype(str).str.strip().tolist()
            return [f for f in feats if f and f.lower() != "nan"]
    feats = fdf.iloc[:, 0].astype(str).str.strip().tolist()
    return [f for f in feats if f and f.lower() != "nan"]

features_used = {
    "FULL": load_feature_list(p_full),
    "CLINICAL": load_feature_list(p_clin),
    "BIOMARKERS": load_feature_list(p_bio),
}

print("Governed feature counts (from S9 lists):", {k: len(v) for k, v in features_used.items()})

pd.Series(features_used["FULL"]).to_csv(TAB_DIR / "S10__features_used_FULL_snapshot.csv", index=False, header=["feature"])
pd.Series(features_used["CLINICAL"]).to_csv(TAB_DIR / "S10__features_used_CLINICAL_snapshot.csv", index=False, header=["feature"])
pd.Series(features_used["BIOMARKERS"]).to_csv(TAB_DIR / "S10__features_used_BIOMARKERS_snapshot.csv", index=False, header=["feature"])


Loaded: /content/processed_dataset.csv | shape=(152, 117)
Feature lists:
 - /content/features_used_FULL.csv
 - /content/features_used_CLINICAL.csv
 - /content/features_used_BIOMARKERS.csv
S9 split: /content/S9_locked_split_indices.csv
Audit nonnumeric: None
Governed feature counts (from S9 lists): {'FULL': 69, 'CLINICAL': 58, 'BIOMARKERS': 10}


## Target, class filtering, TRAIN-only with S9

In [3]:
TARGET_COL = "typ_zawalu"
if TARGET_COL not in df_raw.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in processed_dataset.csv")

labels = df_raw[TARGET_COL].astype(str).str.strip()
allowed = {"STEMI", "NSTEMI"}

df = df_raw.loc[labels.isin(allowed)].copy()
y_all = (df[TARGET_COL].astype(str).str.strip() == "STEMI").astype(int).values

print("TARGET_COL:", TARGET_COL)
print("Task: STEMI vs NSTEMI (positive=STEMI)")
print(pd.Series(y_all).value_counts())
print("Positive rate:", float(np.mean(y_all)))
print("df filtered shape:", df.shape)

NONNUMERIC_FEATURES = set()
if audit_df is not None and len(audit_df) > 0:
    cand = ["feature", "feature_name", "column", "variable", "var", "name"]
    col = next((c for c in cand if c in audit_df.columns), None)
    if col is None:
        col = audit_df.columns[0]
    NONNUMERIC_FEATURES = set(audit_df[col].astype(str).tolist())

print("Non-numeric features flagged:", len(NONNUMERIC_FEATURES))

def _infer_train_indices_from_s9(path: Path, n_rows: int) -> np.ndarray:
    df_split = pd.read_csv(path)

    split_cols = [c for c in df_split.columns if str(c).lower() in {"split","set","subset","group","partition"}]
    idx_cols = [c for c in df_split.columns if str(c).lower() in {"index","idx","row","row_id","rowid","id"}]

    if split_cols and idx_cols:
        split_c = split_cols[0]
        idx_c = idx_cols[0]
        mask = df_split[split_c].astype(str).str.upper().str.contains("TRAIN")
        vals = df_split.loc[mask, idx_c].dropna().astype(int).values
        if len(vals):
            vals = np.unique(vals)
            vals = vals[(vals >= 0) & (vals < n_rows)]
            return vals

    for c in df_split.columns:
        s = pd.to_numeric(df_split[c], errors="coerce").dropna()
        if len(s) and s.min() >= 0 and s.max() < n_rows:
            return np.unique(s.astype(int).values)

    return np.arange(n_rows)

if USE_TRAIN_ONLY_FROM_S9_SPLIT:
    train_idx = _infer_train_indices_from_s9(S9_SPLIT_PATH, len(df))
    if len(train_idx) == 0:
        raise ValueError("Empty TRAIN indices inferred from S9 split file.")
    if train_idx.max() >= len(df):
        raise ValueError("S9 TRAIN indices out of range for filtered df. Check filtering consistency.")
    print(f"Using TRAIN subset from S9 split: n={len(train_idx)} / {len(df)}")
else:
    train_idx = np.arange(len(df))
    print("Using ALL rows (TRAIN+LOCKED) — not recommended for S10 if you want S9 consistency.")

df_use = df.iloc[train_idx].copy()
y = y_all[train_idx].copy()

print("df_use shape:", df_use.shape, "| y shape:", y.shape)
print("Positive rate (used):", float(np.mean(y)))


TARGET_COL: typ_zawalu
Task: STEMI vs NSTEMI (positive=STEMI)
0    32
1    25
Name: count, dtype: int64
Positive rate: 0.43859649122807015
df filtered shape: (57, 117)
Non-numeric features flagged: 0
Using TRAIN subset from S9 split: n=45 / 57
df_use shape: (45, 117) | y shape: (45,)
Positive rate (used): 0.4444444444444444


## X matrix construction (S9-style) + variant diagnostics

In [4]:
YES_SET = {"yes","y","true","t","1","tak","on","present","positive","pos"}
NO_SET  = {"no","n","false","f","0","nie","off","absent","negative","neg"}

def _normalize_str_series(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.strip()
             .str.lower()
             .str.replace(",", ".", regex=False))

def _try_map_binary_object(col: pd.Series) -> tuple[pd.Series, bool]:
    norm = _normalize_str_series(col)
    uniq = sorted(norm.dropna().unique().tolist())
    if len(uniq) == 0:
        return col, False
    if len(set(uniq)) <= 2 and all(u in (YES_SET | NO_SET) for u in uniq):
        def map_one(v):
            if pd.isna(v): return np.nan
            vv = str(v).strip().lower()
            if vv in YES_SET: return 1
            if vv in NO_SET:  return 0
            return np.nan
        return norm.map(map_one).astype(float), True
    return col, False

def build_matrix_s9_style(df_in: pd.DataFrame, feats: list[str], variant_name: str):
    feats = [f for f in feats if f in df_in.columns and f != TARGET_COL]
    feats = [f for f in feats if f not in NONNUMERIC_FEATURES]

    Xdf = df_in[feats].copy()

    recoded = 0
    for c in Xdf.columns:
        if Xdf[c].dtype == "object" or str(Xdf[c].dtype).startswith("bool"):
            mapped, ok = _try_map_binary_object(Xdf[c])
            if ok:
                Xdf[c] = mapped
                recoded += 1

    for c in Xdf.columns:
        if Xdf[c].dtype == "object":
            Xdf[c] = _normalize_str_series(Xdf[c])
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")

    all_nan = Xdf.columns[Xdf.isna().mean() == 1.0].tolist()
    constant = Xdf.columns[Xdf.nunique(dropna=True) <= 1].tolist()
    dropped = sorted(set(all_nan + constant))

    if dropped:
        pd.DataFrame({"feature": dropped}).to_csv(
            TAB_DIR / f"S10_{variant_name}__dropped_empty_or_constant.csv", index=False
        )
        Xdf = Xdf.drop(columns=dropped)

    diag = {
        "variant": variant_name,
        "n_governed_present": int(len(feats)),
        "n_recoded_binary": int(recoded),
        "n_dropped_empty_or_constant": int(len(dropped)),
        "n_used_matrix": int(Xdf.shape[1]),
    }
    return Xdf.values, np.array(Xdf.columns.tolist()), diag

variants = {}
diag_rows = []

for vname in ["FULL", "CLINICAL", "BIOMARKERS"]:
    feats_list = features_used[vname]
    Xv, feats_final, diag = build_matrix_s9_style(df_use, feats_list, vname)

    if Xv.shape[1] < 2:
        print(f"WARNING: {vname} has <2 usable predictors after governance; skipping.")
        continue

    variants[vname] = (Xv, feats_final)
    diag_rows.append(diag)

    pd.Series(list(feats_final)).to_csv(
        TAB_DIR / f"S10_{vname}__features_used_final.csv", index=False, header=["feature"]
    )
    print(f"{vname}: {Xv.shape} (n_features={len(feats_final)})")

diag_df = pd.DataFrame(diag_rows).sort_values("variant").reset_index(drop=True)
diag_path = TAB_DIR / "S10_variant_matrix_diagnostics.csv"
diag_df.to_csv(diag_path, index=False)
print("Saved:", diag_path)

try:
    display(diag_df)
except NameError:
    print(diag_df)

if len(variants) == 0:
    raise RuntimeError("No variants have >=2 usable predictors. Check feature lists and dataset.")


FULL: (45, 53) (n_features=53)
CLINICAL: (45, 42) (n_features=42)
BIOMARKERS: (45, 10) (n_features=10)
Saved: /content/results/S10_permutation_diagnostic/tables/S10_variant_matrix_diagnostics.csv


,variant,n_governed_present,n_recoded_binary,n_dropped_empty_or_constant,n_used_matrix
0,BIOMARKERS,10,0,0,10
1,CLINICAL,58,20,16,42
2,FULL,69,20,16,53


## CV + permutation parameters (batching)

In [5]:
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

N_PERM = 500
CHECKPOINT_EVERY = 10


MAX_PERMS_PER_RUN = 50   # 25 / 50 / 100

USE_RFE = False
N_FEATURES_TO_SELECT = 10


OUTER_SPLITS_MODE = "fixed"


N_JOBS_GRID_DEFAULT = 1

print("cv_outer:", cv_outer)
print("cv_inner:", cv_inner)
print("N_PERM:", N_PERM, "| CHECKPOINT_EVERY:", CHECKPOINT_EVERY)
print("MAX_PERMS_PER_RUN:", MAX_PERMS_PER_RUN)
print("OUTER_SPLITS_MODE:", OUTER_SPLITS_MODE)
print("N_JOBS_GRID_DEFAULT:", N_JOBS_GRID_DEFAULT)


cv_outer: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
cv_inner: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
N_PERM: 500 | CHECKPOINT_EVERY: 10
MAX_PERMS_PER_RUN: 50
OUTER_SPLITS_MODE: fixed
N_JOBS_GRID_DEFAULT: 1


## Models + grids (SVM fast, RF grid reduced)


In [6]:
REVISED_MODELS = {
    "LR_L1": LogisticRegression(penalty="l1", solver="saga", max_iter=8000, random_state=RANDOM_STATE),
    "LR_ELNET": LogisticRegression(penalty="elasticnet", solver="saga", max_iter=8000, random_state=RANDOM_STATE),

    "SVM_REG": SVC(probability=False, random_state=RANDOM_STATE),
    "RF_SHALLOW": RandomForestClassifier(random_state=RANDOM_STATE),
}

REVISED_GRIDS = {
    "LR_L1": {"clf__C": np.logspace(-3, 2, 10)},
    "LR_ELNET": {"clf__C": np.logspace(-3, 2, 10), "clf__l1_ratio": [0.1, 0.5, 0.9]},
    "SVM_REG": {"clf__C": np.logspace(-3, 2, 10), "clf__kernel": ["linear","rbf"], "clf__gamma": ["scale"]},


    "RF_SHALLOW": {
        "clf__n_estimators": [300],
        "clf__max_depth": [2,3,4],
        "clf__min_samples_leaf": [5,10],
        "clf__max_features": ["sqrt"],
    },
}

def make_revised_pipeline(model_key: str, use_rfe: bool = False, n_features_to_select: int = 10):
    pre_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_key in ["LR_L1","LR_ELNET","SVM_REG"]:
        pre_steps.append(("scaler", StandardScaler()))
    preprocess = Pipeline(pre_steps)

    steps = [("preprocess", preprocess)]

    if use_rfe:
        rfe_est = LogisticRegression(max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)
        steps.append(("rfe", RFE(estimator=rfe_est, n_features_to_select=n_features_to_select)))

    steps.append(("clf", REVISED_MODELS[model_key]))
    return Pipeline(steps)

print("Models:", list(REVISED_MODELS.keys()))


Models: ['LR_L1', 'LR_ELNET', 'SVM_REG', 'RF_SHALLOW']


## Features: metrics (safe), nested CV

In [7]:
def _has_two_classes(y_vec: np.ndarray) -> bool:
    y_u = np.unique(y_vec[~pd.isna(y_vec)])
    return len(y_u) >= 2

def nested_cv_metrics_with_outer_splits(
    X, y_vec, outer_splits, variant_name, use_rfe=False,
    save_fold_details_path=None,
    n_jobs_grid=1
):
    rows = []
    fold_details = []

    for model_key in REVISED_MODELS.keys():
        aucs, mccs = [], []

        n_select_eff = None
        if use_rfe:
            n_select_eff = min(N_FEATURES_TO_SELECT, X.shape[1]-1)
            use_rfe_eff = n_select_eff >= 2
        else:
            use_rfe_eff = False

        for fold_id, (tr, te) in enumerate(outer_splits, start=1):
            X_tr, X_te = X[tr], X[te]
            y_tr, y_te = y_vec[tr], y_vec[te]

            if not _has_two_classes(y_tr):
                aucs.append(np.nan)
                mccs.append(np.nan)
                fold_details.append({
                    "variant": variant_name,
                    "model": model_key + ("_RFE" if use_rfe_eff else ""),
                    "fold_id": fold_id,
                    "best_score_inner_cv": np.nan,
                    "best_params": None,
                    "use_rfe_effective": bool(use_rfe_eff),
                    "n_select_effective": int(n_select_eff) if use_rfe_eff else None,
                    "error": "TRAIN_HAS_ONE_CLASS"
                })
                continue

            pipe = make_revised_pipeline(
                model_key, use_rfe=use_rfe_eff, n_features_to_select=(n_select_eff or N_FEATURES_TO_SELECT)
            )

            grid = GridSearchCV(
                pipe,
                REVISED_GRIDS[model_key],
                scoring="roc_auc",
                cv=cv_inner,
                n_jobs=n_jobs_grid,
                refit=True,
                error_score=np.nan
            )

            try:
                grid.fit(X_tr, y_tr)
                best_model = grid.best_estimator_


                if hasattr(best_model, "predict_proba"):
                    scores = best_model.predict_proba(X_te)[:, 1]
                    thr = 0.5
                elif hasattr(best_model, "decision_function"):
                    scores = best_model.decision_function(X_te)
                    thr = 0.0
                else:
                    scores = best_model.predict(X_te)
                    thr = 0.5

                if _has_two_classes(y_te):
                    auc_val = float(roc_auc_score(y_te, scores))
                else:
                    auc_val = np.nan

                preds = (scores >= thr).astype(int)
                mcc_val = float(matthews_corrcoef(y_te, preds)) if _has_two_classes(y_te) else np.nan

                aucs.append(auc_val)
                mccs.append(mcc_val)

                fold_details.append({
                    "variant": variant_name,
                    "model": model_key + ("_RFE" if use_rfe_eff else ""),
                    "fold_id": fold_id,
                    "best_score_inner_cv": float(grid.best_score_) if pd.notna(grid.best_score_) else np.nan,
                    "best_params": json.dumps(grid.best_params_, ensure_ascii=False) if hasattr(grid, "best_params_") else None,
                    "use_rfe_effective": bool(use_rfe_eff),
                    "n_select_effective": int(n_select_eff) if use_rfe_eff else None,
                    "error": None
                })

            except ValueError as e:
                aucs.append(np.nan)
                mccs.append(np.nan)
                fold_details.append({
                    "variant": variant_name,
                    "model": model_key + ("_RFE" if use_rfe_eff else ""),
                    "fold_id": fold_id,
                    "best_score_inner_cv": np.nan,
                    "best_params": None,
                    "use_rfe_effective": bool(use_rfe_eff),
                    "n_select_effective": int(n_select_eff) if use_rfe_eff else None,
                    "error": f"ValueError: {str(e)}"
                })
                continue

        auc_mean = float(np.nanmean(aucs)) if np.any(~np.isnan(aucs)) else np.nan
        mcc_mean = float(np.nanmean(mccs)) if np.any(~np.isnan(mccs)) else np.nan

        auc_sd = float(np.nanstd(aucs, ddof=1)) if np.sum(~np.isnan(aucs)) > 1 else np.nan
        mcc_sd = float(np.nanstd(mccs, ddof=1)) if np.sum(~np.isnan(mccs)) > 1 else np.nan

        rows.append({
            "variant": variant_name,
            "model": model_key + ("_RFE" if use_rfe_eff else ""),
            "auc_mean": auc_mean,
            "auc_sd": auc_sd,
            "mcc_mean": mcc_mean,
            "mcc_sd": mcc_sd,
            "use_rfe_effective": bool(use_rfe_eff),
            "n_select_effective": int(n_select_eff) if use_rfe_eff else None
        })

    if save_fold_details_path is not None:
        pd.DataFrame(fold_details).to_csv(save_fold_details_path, index=False)

    return pd.DataFrame(rows)


## Functions: permutations (recording after each perm + batch stop), summary

In [8]:
def permutation_diagnostic_for_variant(
    X, y_true, variant_name, n_perm, seed, use_rfe=False,
    outdir=None, checkpoint_every=10,
    n_jobs_grid=1,
    max_perms_per_run=50
):
    assert outdir is not None, "Pass outdir=TAB_DIR"
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    tag = f"{variant_name}{'_RFE' if use_rfe else ''}"
    true_path = outdir / f"S10_{tag}__TRUE.csv"
    perm_path = outdir / f"S10_{tag}__PERM.csv"
    all_path  = outdir / f"S10_{tag}__ALL.csv"
    fold_details_true_path = outdir / f"S10_{tag}__FOLD_DETAILS_TRUE.csv"

    rng = np.random.default_rng(seed)

    outer_splits_true = list(cv_outer.split(X, y_true))


    if not true_path.exists():
        df_true = nested_cv_metrics_with_outer_splits(
            X, y_true, outer_splits_true, variant_name, use_rfe=use_rfe,
            save_fold_details_path=fold_details_true_path,
            n_jobs_grid=n_jobs_grid
        ).assign(label_type="true", perm_id=-1)
        df_true.to_csv(true_path, index=False)
        print(f"[{variant_name}] TRUE saved -> {true_path.name}")
    else:
        df_true = pd.read_csv(true_path)
        print(f"[{variant_name}] TRUE exists -> {true_path.name}")


    done = set()
    if perm_path.exists():
        df_prev = pd.read_csv(perm_path)
        done = set(df_prev["perm_id"].astype(int).unique().tolist())
        print(f"[{variant_name}] Resume: {len(done)} perms already saved -> {perm_path.name}")

    completed_now = 0
    last_saved = None

    for p in range(n_perm):
        if p in done:
            continue

        y_perm = rng.permutation(y_true)

        if OUTER_SPLITS_MODE == "fixed":
            outer_splits_perm = outer_splits_true
        elif OUTER_SPLITS_MODE == "permutation":
            outer_splits_perm = list(cv_outer.split(X, y_perm))
        else:
            raise ValueError("OUTER_SPLITS_MODE must be 'fixed' or 'permutation'")

        df_p = nested_cv_metrics_with_outer_splits(
            X, y_perm, outer_splits_perm, variant_name, use_rfe=use_rfe,
            save_fold_details_path=None,
            n_jobs_grid=n_jobs_grid
        ).assign(label_type="permuted", perm_id=p)


        header = not perm_path.exists()
        df_p.to_csv(perm_path, mode="a", header=header, index=False)

        completed_now += 1
        last_saved = p

        if completed_now % checkpoint_every == 0:
            print(f"[{variant_name}] checkpoint saved (latest perm_id={last_saved})")


        if completed_now >= max_perms_per_run:
            print(f"[{variant_name}] STOP after batch={max_perms_per_run} new perms (latest perm_id={last_saved})")
            break

    df_perm = pd.read_csv(perm_path) if perm_path.exists() else pd.DataFrame()
    df_all = pd.concat([df_true, df_perm], ignore_index=True) if len(df_perm) else df_true.copy()
    df_all.to_csv(all_path, index=False)

    return df_true, df_perm, df_all


def permutation_summary(df_true, df_perm, metric="auc_mean"):
    out = []
    for (variant, model), g_true in df_true.groupby(["variant", "model"]):
        true_val = float(g_true[metric].iloc[0])

        perm_vals = df_perm.loc[
            (df_perm["variant"] == variant) & (df_perm["model"] == model),
            metric
        ].values.astype(float)

        perm_vals = perm_vals[~np.isnan(perm_vals)]

        if len(perm_vals) == 0:
            out.append({
                "variant": variant,
                "model": model,
                f"{metric}_true": true_val,
                f"{metric}_p_value": np.nan,
                f"{metric}_n_perm_used": 0
            })
            continue

        p = (1.0 + float(np.sum(perm_vals >= true_val))) / (1.0 + float(len(perm_vals)))

        out.append({
            "variant": variant,
            "model": model,
            f"{metric}_true": true_val,
            f"{metric}_perm_mean": float(np.mean(perm_vals)),
            f"{metric}_perm_sd": float(np.std(perm_vals, ddof=1)) if len(perm_vals) > 1 else np.nan,
            f"{metric}_perm_p95": float(np.quantile(perm_vals, 0.95)),
            f"{metric}_perm_p99": float(np.quantile(perm_vals, 0.99)),
            f"{metric}_p_value": float(p),
            f"{metric}_n_perm_used": int(len(perm_vals)),
        })
    return pd.DataFrame(out)


## Run variants + master CSV + log

In [9]:
all_true = []
all_perm = []
all_all = []
all_summary = []

variant_names_run = [v for v in ["FULL", "CLINICAL", "BIOMARKERS"] if v in variants]
print("Will run variants in order:", variant_names_run)

for vname in variant_names_run:
    Xv, feats = variants[vname]

    print("\n==============================")
    print("Permutation diagnostic:", vname)
    print("==============================")
    print("X shape:", Xv.shape, "| y shape:", y.shape)

    df_true_v, df_perm_v, df_all_v = permutation_diagnostic_for_variant(
        Xv, y, vname,
        n_perm=N_PERM,
        seed=RANDOM_STATE,
        use_rfe=USE_RFE,
        outdir=TAB_DIR,
        checkpoint_every=CHECKPOINT_EVERY,
        n_jobs_grid=N_JOBS_GRID_DEFAULT,
        max_perms_per_run=MAX_PERMS_PER_RUN
    )

    summ_auc = permutation_summary(df_true_v, df_perm_v, metric="auc_mean")
    summ_mcc = permutation_summary(df_true_v, df_perm_v, metric="mcc_mean")
    summ = pd.merge(summ_auc, summ_mcc, on=["variant", "model"], how="inner", suffixes=("_auc", "_mcc"))

    tag = f"{vname}{'_RFE' if USE_RFE else ''}"
    summ_path = TAB_DIR / f"S10_{tag}__SUMMARY.csv"
    summ.to_csv(summ_path, index=False)
    print(f"[{vname}] summary saved -> {summ_path.name}")

    all_true.append(df_true_v)
    all_perm.append(df_perm_v)
    all_all.append(df_all_v)
    all_summary.append(summ)

df_true_all = pd.concat(all_true, ignore_index=True) if all_true else pd.DataFrame()
df_perm_all = pd.concat(all_perm, ignore_index=True) if all_perm else pd.DataFrame()
df_all_all = pd.concat(all_all, ignore_index=True) if all_all else pd.DataFrame()
df_summary_all = pd.concat(all_summary, ignore_index=True) if all_summary else pd.DataFrame()

p_true = TAB_DIR / "S10_MASTER__TRUE.csv"
p_perm = TAB_DIR / "S10_MASTER__PERM.csv"
p_all  = TAB_DIR / "S10_MASTER__ALL.csv"
p_sum  = TAB_DIR / "S10_MASTER__SUMMARY.csv"

df_true_all.to_csv(p_true, index=False)
df_perm_all.to_csv(p_perm, index=False)
df_all_all.to_csv(p_all, index=False)
df_summary_all.to_csv(p_sum, index=False)

run_log = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "random_state": RANDOM_STATE,
    "task": "Permutation diagnostic (nested CV) on TRAIN-only subset from S9",
    "target": {
        "TARGET_COL": TARGET_COL,
        "classes_included": ["STEMI", "NSTEMI"],
        "positive_class": "STEMI",
        "n_samples_after_filtering": int(len(df)),
        "positive_rate_after_filtering": float(np.mean(y_all)),
        "n_samples_used": int(len(df_use)),
        "positive_rate_used": float(np.mean(y)),
    },
    "inputs": {
        "processed_dataset.csv": str(path_processed),
        "features_used_FULL.csv": str(p_full),
        "features_used_CLINICAL.csv": str(p_clin),
        "features_used_BIOMARKERS.csv": str(p_bio),
        "audit_nonnumeric_features.csv": str(path_audit) if path_audit else None,
        "S9_locked_split_indices.csv": str(S9_SPLIT_PATH) if S9_SPLIT_PATH else None,
    },
    "cv": {
        "outer": 5,
        "inner": 5,
        "n_perm": N_PERM,
        "checkpoint_every": CHECKPOINT_EVERY,
        "max_perms_per_run": MAX_PERMS_PER_RUN,
        "outer_splits_mode": OUTER_SPLITS_MODE,
        "outer_splits_fixed_across_true_and_perm": (OUTER_SPLITS_MODE == "fixed")
    },
    "models": list(REVISED_MODELS.keys()),
    "rfe": {"enabled": USE_RFE, "n_features_to_select": N_FEATURES_TO_SELECT},
    "runtime": {
        "python": platform.python_version(),
        "sklearn": sklearn.__version__,
        "n_jobs_grid": N_JOBS_GRID_DEFAULT
    },
    "outputs": {
        "variant_matrix_diagnostics": str(diag_path),
        "master_true": str(p_true),
        "master_perm": str(p_perm),
        "master_all": str(p_all),
        "master_summary": str(p_sum),
    },
    "notes": [
        "S10 uses the same target definition and class filtering as S9.",
        "S10 uses the same governed feature lists as S9 (features_used_*.csv).",
        "If TRAIN-only is enabled, S10 uses exactly the TRAIN indices from S9 (locked test excluded).",
        "Hyperparameter tuning is performed only within the training portion of each outer fold (nested CV).",
        "AUC/MCC are computed with NaN-safe guards when a fold has only one class (common under permutation).",
        "SVM uses decision_function for AUC (probability=False) for stability/speed in Colab.",
        "Permutation results are appended after each permutation and run is batch-limited for Colab stability."
    ]
}

log_path = LOG_DIR / "S10_run_log.json"
log_path.write_text(json.dumps(run_log, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nDONE. Results saved in:", OUT_DIR.resolve())
print("Saved:", log_path)


Will run variants in order: ['FULL', 'CLINICAL', 'BIOMARKERS']

Permutation diagnostic: FULL
X shape: (45, 53) | y shape: (45,)
[FULL] TRUE saved -> S10_FULL__TRUE.csv
[FULL] checkpoint saved (latest perm_id=9)
[FULL] checkpoint saved (latest perm_id=19)
[FULL] checkpoint saved (latest perm_id=29)
[FULL] checkpoint saved (latest perm_id=39)
[FULL] checkpoint saved (latest perm_id=49)
[FULL] STOP after batch=50 new perms (latest perm_id=49)
[FULL] summary saved -> S10_FULL__SUMMARY.csv

Permutation diagnostic: CLINICAL
X shape: (45, 42) | y shape: (45,)
[CLINICAL] TRUE saved -> S10_CLINICAL__TRUE.csv
[CLINICAL] checkpoint saved (latest perm_id=9)
[CLINICAL] checkpoint saved (latest perm_id=19)
[CLINICAL] checkpoint saved (latest perm_id=29)
[CLINICAL] checkpoint saved (latest perm_id=39)
[CLINICAL] checkpoint saved (latest perm_id=49)
[CLINICAL] STOP after batch=50 new perms (latest perm_id=49)
[CLINICAL] summary saved -> S10_CLINICAL__SUMMARY.csv

Permutation diagnostic: BIOMARKERS
X s

/tmp/ipython-input-3272230896.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
